# Conversations Agent

Every conversation with an AIDB agent is logged automatically. This demo turns those logs into a knowledge base and builds a **second agent that analyzes the first ones**, so you can ask questions such as *"What are people asking about?"* and get answers grounded in real usage.

## What you will build

1. A flat table with **one row per conversation**, assembled from the AIDB conversation log.
2. An **AI pipeline** that splits each conversation into chunks and embeds them for semantic search.
3. A **custom SQL tool** that searches those chunks.
4. An **agent** that uses the tool to answer questions about how the other agents are being used.

## Why it is useful

Understanding what users ask, and what answers they receive, is how you find gaps in your data, improve agent instructions, spot misuse, and decide which tools to build next. Because the logs live in Postgres, you can analyze them with the same tools you use for any other data.

*Note: This demo is most useful after completing the other demo notebooks, so that there are agent conversations to analyze.*

## Claude API Key

This demo creates an agent that uses a Claude Sonnet model, so an Anthropic API key is required. Run the next cell and paste your key when prompted. It is held in memory only and passed to the database when the model is registered.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

## Connect to the database

Creates a SQLAlchemy engine for the database and attaches the `%sql` magics to it. `pool_pre_ping=True` checks connections before reuse, and named parameters are enabled so `%%sql` cells can use Python variables such as `:creds`.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Register Claude Sonnet

`aidb.create_model` registers the Claude Sonnet 5 model inside the database under the name `claude_model_agent_conversations`. The agent created later in this notebook uses it for reasoning and for writing its answers. Any existing registration with the same name is deleted first so the notebook can be re-run.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'claude_model_agent_conversations';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

## Agent Messages

AIDB stores conversation activity in two places: `aidb.conversations` and `aidb.conversation_log`. The log has many rows per conversation (one for each step), which is awkward to search. This query reshapes it into **one row per conversation**.

How it works:

- For each conversation, each log entry is turned into text of the form `action_type: contents`.
- The entries are collected in the order they happened (`ORDER BY created_at`) and joined into a single string, separated by `; `.
- The result is saved to a new table, `public.aidb_conversations`, with a `conversation_id` and a `conversation_log` column.

*Note: conversations asked of this demo's own agent (`demo_agent_conversations_log`) are excluded, so the analysis agent does not end up analyzing itself.*

The table is a snapshot. To include newer conversations later, run this cell again and then re-run the pipeline in the next step.

In [ ]:
%%sql
DROP TABLE IF EXISTS public.aidb_conversations;

CREATE TABLE public.aidb_conversations AS
SELECT conversation_id::text AS conversation_id, 
    array_to_string(log, '; ') AS conversation_log
FROM (
    SELECT cl.conversation_id, array_agg(cl.action_type || ': ' || (cl.payload ->> 'contents')::text ORDER BY cl.created_at) AS log
    FROM aidb.conversation_log cl
    LEFT JOIN aidb.agents a ON a.id = cl.agent_id
    WHERE (a.name <> 'demo_agent_conversations_log' OR a.name IS NULL)
    GROUP BY cl.conversation_id
    ) AS sub;

## Create the AI Pipeline

AI Pipelines can ingest, prepare, and vectorize data for semantic retrieval and RAG applications. They integrate pgvector search capabilities and can keep the knowledge base synchronized with your source data automatically.

This pipeline reads the `conversation_log` column of `public.aidb_conversations` and has two steps:

1. **`ChunkText`** splits each conversation into pieces of about 60 words (never more than 120), with 10 words of overlap between neighbouring chunks. Smaller chunks make searches more precise, and the overlap keeps ideas that straddle a boundary from being cut in half. The chunks are saved (`intermediate_destination`) so the matching text can be shown with each search hit.
2. **`KnowledgeBase`** creates a vector embedding for each chunk using the `bge-small-en-v1.5-f16` embedding model.

`auto_processing => 'Disabled'` means the pipeline only runs when asked, and `aidb.run_pipeline` does exactly that. The first statement deletes any previous pipeline of the same name so the cell can be re-run.

In [ ]:
%%sql

DO $$
DECLARE
    v_name text := 'aidb_conversations_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.pipelines WHERE name = v_name) THEN
        PERFORM aidb.delete_pipeline(v_name);
    END IF;

    PERFORM aidb.create_pipeline(
        name           => v_name,
        source         => 'aidb_conversations',
        source_key_column  => 'conversation_id',
        source_data_column => 'conversation_log',
        step_1         => 'ChunkText',
        step_1_options => aidb.chunk_text_config(desired_length => 60, max_length => 120, overlap_length => 10, strategy => 'words')
                          || jsonb_build_object('intermediate_destination', jsonb_build_object('enabled', true)),
        step_2         => 'KnowledgeBase',
        step_2_options => aidb.knowledge_base_config(model => 'bge-small-en-v1.5-f16', data_format => 'Text'),
        auto_processing => 'Disabled'
    );

    PERFORM aidb.run_pipeline(v_name);

END
$$

## Example question

Before involving an agent, you can query the knowledge base directly. `aidb.retrieve_text` takes the knowledge base table (the pipeline name with a `pipeline_` prefix), a natural-language question, and the number of results to return. With `deduplicate => false` it can return several chunks from the same conversation.

Each row shows the conversation `key`, a `distance` that indicates how close the chunk is to the question, and the matching chunk text. Notice that the search matches on *meaning*, not on shared keywords.

As before, similarity search is useful, but the real power comes from creating a **custom tool** and letting an agent ask questions about your data.

In [ ]:
%%sql
SELECT key,
       part_ids,
       distance::numeric(5,3) AS dist,
       intermediate_steps -> 0 ->> 'value' AS chunk_text
FROM aidb.retrieve_text('public.pipeline_aidb_conversations_kb', --
                        'what topics are being asked', --question
                        8, --distance
                        deduplicate => false)
ORDER BY 3 DESC;

## Create Custom Tool

Tools are how agents get information and take actions. A **SQL tool** wraps a SQL statement so an agent can call it with parameters.

- `description` is what the agent reads to decide *when* to use the tool, so write it as an instruction to the agent.
- `${query}` and `${top_k}` are placeholders that the agent fills in. They are declared in `params`, each with a type and a short description.
- `read_only => true` marks the tool as safe because it never modifies data.

This tool runs the same `aidb.retrieve_text` search you just ran by hand, but lets the agent choose the search phrase and how many results it wants.

In [ ]:
%%sql
DO $do$

DECLARE
    v_name text := 'search_conversations_log';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.sql_tool_registry WHERE name = v_name) THEN
        PERFORM aidb.delete_tool(v_name);
    END IF;
    
    PERFORM aidb.create_sql_tool(
        name          => 'search_conversations_log',
        description   => 'Semantic search over AIDB Conversation Logs. Returns the most relevant conversations.',
        sql_statement => $$
            SELECT key AS source_file,
                   distance::numeric(5,3) AS distance,
                   intermediate_steps -> 0 ->> 'value' AS passage
            FROM aidb.retrieve_text('public.pipeline_aidb_conversations_kb'::text, ${query}::text, ${top_k}::int, deduplicate => false)
        $$,
        params        => aidb.tool_params(
            aidb.tool_param('query', 'TEXT', 'The question or search phrase.'),
            aidb.tool_param('top_k', 'INT', 'Number of passages to return, e.g. 6.')
        ),
        read_only     => true
    );
END
$do$;

## Create new Agent for Conversation Log Tool

This agent has a single tool, `search_conversations_log`, and instructions that tell it to rely on that tool for its answers. Giving an agent only the tools it needs keeps it focused and predictable. It uses the model registered earlier in this notebook.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'demo_agent_conversations_log';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;
    
    PERFORM aidb.create_agent(v_name, 
        'You are a helpful, read-only database assistant. Use search_conversations_log to find the answers to questions.',
        'claude_model_agent_conversations',
        tools => ARRAY['search_conversations_log']);
END
$do$;

## Ask the Agent

This cell defines a small helper, `ask()`, that wraps `aidb.agent_converse` so the output is easier to read:

- It sends your question to the named agent and renders the reply as formatted Markdown.
- If the agent returns an error, it displays the error instead.
- It remembers the last `conversation_id`, so calling `ask(..., continue_conv=True)` continues the same conversation.
- Passing `debug=True` asks AIDB for extra detail about how the agent reached its answer.

The cell then asks the question *"What are the most common questions asked?"*. The agent will search the conversation log, retrieve the most relevant passages, and summarize the themes. The answer depends on which other notebooks you have already run.

In [ ]:
from sqlalchemy import text
from IPython.display import Markdown, display
import pandas as pd

def ask(agent, question, conversation_id=None):
    """Ask the agent, render the answer, return (message, conversation_id).
    Pass a conversation_id from an earlier call to ask a follow-up."""
    with conn.begin() as c:
        df = pd.read_sql(
            text("""SELECT * FROM aidb.agent_converse(
                        CAST(:agent AS text), CAST(:q AS text),
                        conversation_id => CAST(:cid AS text))"""),
            c, params={"agent": agent, "q": question, "cid": conversation_id})
    row = df.iloc[0]
    if row["error"]:
        display(Markdown(f"**Agent error:** {row['error']}"))
    else:
        display(Markdown(row["message"]))
    # On an error the function returns a NULL id, so keep the one passed in
    new_id = row["conversation_id"]
    return row["message"], (conversation_id if pd.isna(new_id) else new_id)

AGENT = "demo_agent_conversations_log"

_, conversation_id = ask(AGENT, "What are the most common questions asked?")

## Recap

You turned AIDB's own conversation logs into a searchable knowledge base and put an agent on top of it. The same pattern of **table, pipeline, SQL tool, agent** works for any text you keep in Postgres, such as support tickets, call notes, or incident reports.

**Ideas to try next**

- Ask which questions the other agents struggled to answer.
- Re-run the *Agent Messages* and pipeline cells after more conversations have happened to refresh the analysis.